# Day 3 - Model Tuning and Bias-Variance

Tune the four classical candidates with training-only cross-validation, inspect train/validation recall and the learning curve, then evaluate and save the selected model. The fixed holdout is created here and used only at the end.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

df = pd.read_csv(PROJECT_ROOT / "data" / "churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].str.strip(),
    errors="coerce"
)

# Replace missing TotalCharges with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Remove customerID because it is only an identifier
df.drop(columns=["customerID"], inplace=True)

# Encode target variable
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Verify
print("Missing values:", df.isnull().sum().sum())
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Churn dtype:", df["Churn"].dtype)
print("Dataset shape:", df.shape)

Missing values: 0
TotalCharges dtype: float64
Churn dtype: int64
Dataset shape: (7043, 20)


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_18396\1707822974.py:11: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["customerID"], inplace=True)


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training samples: 5634
Testing samples: 1409

Training target distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [4]:
numeric_features = X.select_dtypes(include="number").columns.tolist()

categorical_features = X.select_dtypes(
    exclude="number"
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Identify numerical and categorical features
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Combine preprocessing and classifier
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [6]:
from sklearn.model_selection import GridSearchCV, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Candidate estimators and compact search grids
day3_candidates = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=42),
        {"classifier__C": [0.1, 1, 10],
         "classifier__class_weight": [None, "balanced"]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"classifier__max_depth": [None, 5, 10, 20],
         "classifier__min_samples_leaf": [1, 5, 10],
         "classifier__class_weight": [None, "balanced"]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42, n_jobs=1),
        {"classifier__n_estimators": [100, 250],
         "classifier__max_depth": [None, 10, 20],
         "classifier__min_samples_leaf": [1, 5],
         "classifier__class_weight": [None, "balanced"]},
    ),
    "SVM": (
        SVC(random_state=42),
        {"classifier__C": [0.1, 1, 10],
         "classifier__kernel": ["rbf"],
         "classifier__class_weight": [None, "balanced"]},
    ),
}


In [7]:
# GridSearchCV uses the training partition only; the test partition stays untouched.
day3_searches = {}
day3_rows = []
for name, (classifier, param_grid) in day3_candidates.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier),
    ])
    search = GridSearchCV(
        pipeline, param_grid, scoring="recall", cv=5,
        return_train_score=True, n_jobs=1, refit=True
    )
    print(f"Tuning {name}...")
    search.fit(X_train, y_train)
    day3_searches[name] = search
    best_index = search.best_index_
    day3_rows.append({
        "Model": name,
        "Mean train recall": search.cv_results_["mean_train_score"][best_index],
        "Mean CV recall": search.best_score_,
        "Train-CV gap": (
            search.cv_results_["mean_train_score"][best_index] - search.best_score_
        ),
        "Best parameters": search.best_params_,
    })

day3_results = pd.DataFrame(day3_rows).sort_values(
    "Mean CV recall", ascending=False
).reset_index(drop=True)
display(day3_results.round(4))


Tuning Logistic Regression...
Tuning Decision Tree...
Tuning Random Forest...
Tuning SVM...


,Model,Mean train recall,Mean CV recall,Train-CV gap,Best parameters
0,Logistic Regression,0.8052,0.8040,0.0012,"{'classifier__C': 1, 'classifier__class_weight..."
1,SVM,0.8087,0.7967,0.0120,"{'classifier__C': 0.1, 'classifier__class_weig..."
2,Decision Tree,0.8227,0.7846,0.0381,"{'classifier__class_weight': 'balanced', 'clas..."
3,Random Forest,0.8911,0.7732,0.1179,"{'classifier__class_weight': 'balanced', 'clas..."


In [8]:
# Inspect train and validation recall as the training data grows.
day3_champion_name = max(
    day3_searches, key=lambda name: day3_searches[name].best_score_
)
day3_champion = day3_searches[day3_champion_name].best_estimator_

train_sizes, train_scores, validation_scores = learning_curve(
    day3_champion, X_train, y_train, cv=5, scoring="recall",
    train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0], n_jobs=1
)
day3_learning_curve = pd.DataFrame({
    "Training rows per fold": train_sizes,
    "Mean train recall": train_scores.mean(axis=1),
    "Mean validation recall": validation_scores.mean(axis=1),
})
display(day3_learning_curve.round(4))


,Training rows per fold,Mean train recall,Mean validation recall
0,901,0.8196,0.7953
1,1802,0.8229,0.8033
2,2704,0.8216,0.7987
3,3605,0.8140,0.8027
4,4507,0.8050,0.8040


In [9]:
# Evaluate the selected model on the original test set and persist the complete pipeline.
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from pathlib import Path
import joblib

# Keep the saved service artifact compatible with predict_proba if SVM wins.
if day3_champion_name == "SVM":
    day3_champion.set_params(
        classifier=clone(day3_champion.named_steps["classifier"]).set_params(probability=True)
    )
    day3_champion.fit(X_train, y_train)

day3_test_predictions = day3_champion.predict(X_test)
if hasattr(day3_champion, "predict_proba"):
    day3_test_scores = day3_champion.predict_proba(X_test)[:, 1]
else:
    day3_test_scores = day3_champion.decision_function(X_test)

day3_test_metrics = {
    "Accuracy": accuracy_score(y_test, day3_test_predictions),
    "Precision": precision_score(y_test, day3_test_predictions),
    "Recall": recall_score(y_test, day3_test_predictions),
    "ROC-AUC": roc_auc_score(y_test, day3_test_scores),
}
print("Day 3 champion:", day3_champion_name)
print("Selected parameters:", day3_searches[day3_champion_name].best_params_)
print("Mean CV recall:", round(day3_searches[day3_champion_name].best_score_, 4))
display(pd.DataFrame([day3_test_metrics]).round(4))

day3_model_path = PROJECT_ROOT / "models" / "churn_v3.joblib"
day3_model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(day3_champion, day3_model_path)
print("Saved champion:", day3_model_path)


Day 3 champion: Logistic Regression
Selected parameters: {'classifier__C': 1, 'classifier__class_weight': 'balanced'}
Mean CV recall: 0.804


,Accuracy,Precision,Recall,ROC-AUC
0,0.7381,0.5043,0.7834,0.8416


Saved champion: C:\Users\Abhishek\Desktop\RetailMind-AI\models\churn_v3.joblib


## Day 3 interpretation

The tuned Logistic Regression was selected by its highest mean 5-fold validation recall. Its small train-validation gap indicates limited variance for recall. The learning curve converges near 0.80 and changes little at the largest training sizes, so simply adding more rows is unlikely to make a large difference with the current features. The final test metrics show the tradeoff: recall improves substantially over the Day 1 baseline, while precision and accuracy are lower.

The complete preprocessing and classifier pipeline is saved as `models/churn_v3.joblib`. Detailed recorded results are in [the Day 3 report](../docs/day3.md).


### Recorded Day 3 results

The completed search selected **Logistic Regression** (`C=1`, `class_weight="balanced"`) with mean 5-fold CV recall of **0.8040**. Its mean training recall was 0.8052, a train-validation gap of 0.0012.

| Candidate | Mean CV recall | Train-CV gap |
|-|-:|-:|
| Logistic Regression | **0.8040** | 0.0012 |
| SVM | 0.7967 | 0.0120 |
| Decision Tree | 0.7846 | 0.0381 |
| Random Forest | 0.7732 | 0.1179 |

On the original holdout set, the tuned champion achieved **73.81% accuracy, 50.43% precision, 78.34% recall, and 84.16% ROC-AUC**. The recall gain comes with lower precision than the Day 1 baseline. Learning-curve validation recall rose from 0.7953 to 0.8040 as training size increased, then flattened; train and validation recall converged, showing little variance for this metric and limited expected benefit from more data alone.
